In [ ]:
import numpy as np
import matplotlib.pyplot as plt


P_real_m5 = np.zeros((5, 5, 5))

P_real_m5[0] = np.array([
    [0.947, 0.044, 0.008, 0.001, 0.00],
    [0.954, 0.04, 0.006, 0.00, 0.00],
    [0.938, 0.051, 0.01, 0.001, 0.00],
    [0.911, 0.07, 0.016, 0.003, 0.00],
    [0.919, 0.063, 0.009, 0.009, 0.00]
])

P_real_m5[1] = np.array([
    [0.055, 0.872, 0.071, 0.002, 0.00],
    [0.046, 0.898, 0.055, 0.001, 0.00],
    [0.032, 0.893, 0.073, 0.002, 0.00],
    [0.04, 0.859, 0.097, 0.004, 0.00],
    [0.047, 0.858, 0.092, 0.003, 0.00]
])

P_real_m5[2] = np.array([
    [0.002, 0.057, 0.919, 0.022, 0.00],
    [0.002, 0.052, 0.935, 0.011, 0.00],
    [0.001, 0.038, 0.943, 0.018, 0.00],
    [0.001, 0.039, 0.913, 0.046, 0.001],
    [0.001, 0.043, 0.896, 0.059, 0.001]
])

P_real_m5[3] = np.array([
    [0.001, 0.002, 0.102, 0.876, 0.019],
    [0.001, 0.005, 0.093, 0.886, 0.015],
    [0.00, 0.002, 0.078, 0.905, 0.015],
    [0.001, 0.002, 0.068, 0.898, 0.031],
    [0.001, 0.003, 0.069, 0.884, 0.043]
])

P_real_m5[4] = np.array([
    [0.00, 0.00, 0.011, 0.097, 0.892],
    [0.00, 0.003, 0.009, 0.081, 0.907],
    [0.001, 0.001, 0.006, 0.072, 0.92],
    [0.00, 0.001, 0.006, 0.069, 0.924],
    [0.001, 0.00, 0.005, 0.071, 0.923]
])


print("Проверка нормировки:", np.allclose(P_real_m5.sum(axis=2), 1))

In [ ]:
class EpsilonGreedy:
    def __init__(self, n_arms, epsilon=0.1):
        self.n_arms = n_arms
        self.epsilon = epsilon
        self.Q = np.zeros(n_arms)
        self.N = np.zeros(n_arms)

    def select_arm(self):
        if np.random.random() < self.epsilon:
            return np.random.randint(0, self.n_arms)
        else:
            return np.argmax(self.Q)

    def update(self, chosen_arm, reward):
        self.N[chosen_arm] += 1
        self.Q[chosen_arm] += (reward - self.Q[chosen_arm]) / self.N[chosen_arm]

class UCB1:
    def __init__(self, n_arms):
        self.n_arms = n_arms
        self.Q = np.zeros(n_arms)
        self.N = np.zeros(n_arms)
        self.total_counts = 0

    def select_arm(self):
        for arm in range(self.n_arms):
            if self.N[arm] == 0:
                return arm


        ucb_values = self.Q + np.sqrt(2 * np.log(self.total_counts) / self.N)
        return np.argmax(ucb_values)

    def update(self, chosen_arm, reward):
        self.N[chosen_arm] += 1
        self.total_counts += 1
        self.Q[chosen_arm] += (reward - self.Q[chosen_arm]) / self.N[chosen_arm]

class ThompsonSampling:
    def __init__(self, n_arms, reward_range=(-10, 0)):
        self.n_arms = n_arms
        self.alphas = np.ones(n_arms)
        self.betas = np.ones(n_arms)
        self.reward_min, self.reward_max = reward_range

    def _normalize_reward(self, reward):
        return (reward - self.reward_min) / (self.reward_max - self.reward_min)

    def select_arm(self):
        samples = np.random.beta(self.alphas, self.betas)
        return np.argmax(samples)

    def update(self, chosen_arm, reward):
        p = self._normalize_reward(reward)
        self.alphas[chosen_arm] += p
        self.betas[chosen_arm] += (1 - p)

In [ ]:
def run_scardo_mab_experiment(P, m, n_agents=10, n_iter=500,
                              mab_algorithm='ucb1', epsilon=0.1):

    opinions = np.random.randint(1, m+1, n_agents)
    target_opinion = opinions[0]
    n_arms = n_agents - 1

    if mab_algorithm == 'epsilon_greedy':
        mab = EpsilonGreedy(n_arms, epsilon)
    elif mab_algorithm == 'ucb':
        mab = UCB1(n_arms)
    elif mab_algorithm == 'thompson':
        mab = ThompsonSampling(n_arms, reward_range=(-(m-1), 0))
    else:
        raise ValueError(f"Неизвестный алгоритм: {mab_algorithm}")

    rewards_history = []
    opinion_history = [target_opinion]

    for step in range(n_iter):
        partner_idx = mab.select_arm()
        real_partner_idx = partner_idx if partner_idx < 0 else partner_idx + 1

        partner_opinion = opinions[real_partner_idx]


        s = target_opinion - 1
        l = partner_opinion - 1

        possible_opinions = np.arange(1, m+1)
        new_opinion = np.random.choice(possible_opinions, p=P[s, l, :])

        reward =  -abs(target_opinion - new_opinion)
        rewards_history.append(reward)

        target_opinion = new_opinion
        opinion_history.append(target_opinion)

        mab.update(partner_idx, reward)

    return rewards_history, opinion_history

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats


def run_multiple_simulations(P, m, n_agents, n_iter, n_simulations=200,
                           mab_algorithm='ucb', epsilon=0.1):
    all_rewards = []
    all_opinions = []

    for sim in range(n_simulations):
        rewards, opinions = run_scardo_mab_experiment(
            P=P, m=m, n_agents=n_agents, n_iter=n_iter,
            mab_algorithm=mab_algorithm, epsilon=epsilon
        )
        all_rewards.append(rewards[:n_iter])
        all_opinions.append(opinions[1:n_iter+1])

    return np.array(all_rewards), np.array(all_opinions)


m = 5
n_agents = 5
n_iter = 100
n_simulations = 1000
test_steps = [15, 50, 100]

algorithms = ['epsilon_greedy', 'ucb', 'thompson']
results = {}

print("Запуск множественных симуляций...")
for algo in algorithms:
    rewards, opinions = run_multiple_simulations(
        P=P_real_m5, m=m, n_agents=n_agents, n_iter=n_iter,
        n_simulations=n_simulations, mab_algorithm=algo
    )
    results[algo] = {'rewards': rewards, 'opinions': opinions}


steps = np.arange(n_iter)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
colors = {'epsilon_greedy': 'blue', 'ucb': 'orange', 'thompson': 'green'}


for algo in algorithms:
    cum_rewards = np.cumsum(results[algo]['rewards'], axis=1)
    mean_cum = np.mean(cum_rewards, axis=0)
    se_cum = np.std(cum_rewards, axis=0) / np.sqrt(n_simulations)

    ax1.plot(steps, mean_cum, label=algo, color=colors[algo], linewidth=2)
    ax1.fill_between(steps, mean_cum - se_cum, mean_cum + se_cum,
                     color=colors[algo], alpha=0.2)

ax1.set_xlabel('Шаг'); ax1.set_ylabel('Накопленная награда')
ax1.legend(); ax1.grid(True, alpha=0.3)

for algo in algorithms:
    norm_opinions = (results[algo]['opinions'] - 1) / (m - 1)
    mean_op = np.mean(norm_opinions, axis=0)
    se_op = np.std(norm_opinions, axis=0) / np.sqrt(n_simulations)

    ax2.plot(steps, mean_op, label=algo, color=colors[algo], linewidth=2)
    ax2.fill_between(steps, mean_op - se_op, mean_op + se_op,
                     color=colors[algo], alpha=0.2)

ax2.set_xlabel('Шаг'); ax2.set_ylabel('Среднее мнение [0,1]')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()


import numpy as np
from scipy.stats import mannwhitneyu
print("\n" + "="*80)
print("MANN-WHITNEY U: ПОПАРНОЕ СРАВНЕНИЕ ВСЕХ АЛГОРИТМОВ")
print("="*80)

test_steps = [15, 50, 100]
cum_rewards_all = {algo: np.cumsum(results[algo]['rewards'], axis=1) for algo in algorithms}
algos = ['epsilon_greedy', 'ucb', 'thompson']

print(f"{'Шаг':<4} {'BASELINE':<13} {'vs':<12} {'U':<8} {'p-value':<8} {'r':<6} {'Значимо'}")
print("-"*70)

for step in test_steps:
    step_rewards = {algo: cum_rewards_all[algo][:, step-1] for algo in algorithms}

    # Все возможные пары (каждый по очереди как baseline)
    for i, baseline in enumerate(algos):
        for j, challenger in enumerate(algos):
            if i >= j:  # Избегаем повторов A vs A и A vs B после B vs A
                continue

            stat, p = mannwhitneyu(step_rewards[challenger], step_rewards[baseline],
                                 alternative='greater')  # challenger > baseline

            N = 2 * len(step_rewards[baseline])
            r = abs(stat - (len(step_rewards[baseline])**2)/2) / np.sqrt(N*(N+1)/12)
            sig = '★★★' if p < 0.01 else '★' if p < 0.05 else '-'

            print(f"{step:<4} {baseline:<13} {challenger:<12} {stat:<8.0f} {p:<8.4f} {r:<6.3f} {sig}")

print("\n" + "="*80)
print("ПОЛНАЯ МАТРИЦА СРАВНЕНИЙ (шаг 100):")
print("="*80)

final_rewards = {algo: cum_rewards_all[algo][:, -1] for algo in algorithms}
print(f"{'':<15}", end="")
for baseline in algos:
    print(f"{baseline:^12}", end="")
print()

for challenger in algos:
    print(f"{challenger:<15}", end="")
    for baseline in algos:
        if challenger == baseline:
            print(f"{'-':^12}", end="")
        else:
            stat, p = mannwhitneyu(final_rewards[challenger], final_rewards[baseline],
                                 alternative='greater')
            sig = '★★★' if p < 0.01 else '★' if p < 0.05 else '-'
            print(f"{sig:^12}", end="")
    print()



In [ ]:
def run_multiple_simulations(P, m, n_agents, n_iter, n_simulations=100,
                           mab_algorithm='ucb', epsilon=0.1, true_opinion=None):
    all_rewards = []
    all_opinions = []

    for sim in range(n_simulations):
        if true_opinion is None:
            rewards, opinions = run_scardo_mab_experiment(
                P=P, m=m, n_agents=n_agents, n_iter=n_iter,
                mab_algorithm=mab_algorithm, epsilon=epsilon
            )
        else:
            rewards, opinions = run_scardo_mab_experiment_truth(
                P=P, m=m, n_agents=n_agents, n_iter=n_iter,
                mab_algorithm=mab_algorithm, epsilon=epsilon,
                true_opinion=true_opinion
            )

        all_rewards.append(rewards[:n_iter])
        all_opinions.append(opinions[1:n_iter+1])

    return np.array(all_rewards), np.array(all_opinions)

def run_scardo_mab_experiment_truth(P, m, n_agents=10, n_iter=500,
                                  mab_algorithm='ucb', epsilon=0.1,
                                  true_opinion=3):

    opinions = np.random.randint(1, m+1, n_agents)
    target_opinion = opinions[0]

    n_arms = n_agents - 1
    if mab_algorithm == 'epsilon_greedy':
        mab = EpsilonGreedy(n_arms, epsilon)
    elif mab_algorithm == 'ucb':
        mab = UCB1(n_arms)
    elif mab_algorithm == 'thompson':
        mab = ThompsonSampling(n_arms, reward_range=(-(m-1), 0))

    rewards_history = []
    opinion_history = [target_opinion]

    for step in range(n_iter):
        partner_idx = mab.select_arm()
        real_partner_idx = partner_idx + 1
        partner_opinion = opinions[real_partner_idx]

        s = target_opinion - 1
        l = partner_opinion - 1
        new_opinion = np.random.choice(np.arange(1, m+1), p=P[s, l, :])

        reward = -abs(new_opinion - true_opinion)

        rewards_history.append(reward)
        target_opinion = new_opinion
        opinion_history.append(target_opinion)
        mab.update(partner_idx, reward)

    return rewards_history, opinion_history

m = 5
n_agents = 10
n_iter = 100
true_opinion = 5
n_simulations = 250

algorithms = ['epsilon_greedy', 'ucb', 'thompson']
new_results = {}

for algo in algorithms:
    rewards, opinions = run_multiple_simulations(
        P=P_real_m5, m=m, n_agents=n_agents, n_iter=n_iter,
        n_simulations=n_simulations, mab_algorithm=algo,
        true_opinion=true_opinion
    )
    new_results[algo] = {'rewards': rewards, 'opinions': opinions}


steps = np.arange(n_iter)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
colors = {'epsilon_greedy': 'blue', 'ucb': 'orange', 'thompson': 'green'}


for algo in algorithms:
    cum_rewards = np.cumsum(new_results[algo]['rewards'], axis=1)
    mean_cum = np.mean(cum_rewards, axis=0)
    se_cum = np.std(cum_rewards, axis=0) / np.sqrt(n_simulations)
    ax1.plot(steps, mean_cum, label=algo, color=colors[algo])
    ax1.fill_between(steps, mean_cum-se_cum, mean_cum+se_cum,
                     color=colors[algo], alpha=0.2)

ax1.set_title('Накопленная награда (-|мнение-истина|)')
ax1.set_xlabel('Шаг')
ax1.legend(); ax1.grid(True, alpha=0.3)


for algo in algorithms:
    opinions = new_results[algo]['opinions']
    norm_opinions = (opinions[:, :n_iter] - 1) / (m - 1)
    mean_op = np.mean(norm_opinions, axis=0)
    se_op = np.std(norm_opinions, axis=0) / np.sqrt(n_simulations)

    ax2.plot(steps, mean_op, label=algo, color=colors[algo], linewidth=2)
    ax2.fill_between(steps, mean_op - se_op, mean_op + se_op,
                     color=colors[algo], alpha=0.2)

ax2.set_title('Среднее мнение (true=1)')
ax2.set_xlabel('Шаг')
ax2.set_ylabel('Нормализованное мнение [0,1]')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



print("\n" + "="*60)
print("MANN-WHITNEY U (шаг 100):")
print("="*60)
final_rewards = {algo: np.sum(new_results[algo]['rewards'], axis=1) for algo in algorithms}

for challenger in ['ucb', 'thompson']:
    stat, p = mannwhitneyu(final_rewards[challenger], final_rewards['epsilon_greedy'])
    sig = '★★★' if p < 0.01 else '★' if p < 0.05 else '-'
    print(f"{challenger} vs ε-greedy: U={stat:.0f}, p={p:.4f} {sig}")